In [0]:
%run ../delta_function

In [0]:
# import des bibliothèques
import os
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql import Window
import pyspark.sql.utils;
from pyspark.sql.types import StructType, StringType, IntegerType, TimestampType
from pyspark.sql.functions import concat, lit, col, upper, max, when, row_number, round, length, expr, regexp_extract, explode, to_date, first, udf, current_timestamp, round, regexp_replace, format_string, date_format, trim
from datetime import datetime, timedelta
import numpy as np
from pyspark.sql import SparkSession
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [0]:
try:
    verbose_mode = dbutils.widgets.get("verbose_mode");
except:
    verbose_mode = 'debug'
try:
    current_division = dbutils.widgets.get("division");
except:
    current_division = 'mal'
try:
    current_environment = dbutils.widgets.get("environment");
except:
    current_environment = 'dev'
try:
    execution_mode = dbutils.widgets.get("execution_mode");
except:
    execution_mode = 'update'
try:
    current_project = dbutils.widgets.get("project");
except:
    current_project = 'maite_bi'
try:
    current_production_line = dbutils.widgets.get("production_line");
except:
    current_production_line = 'self_service'

current_catalog = current_division + '_' + current_project + '_' + current_environment;

current_schema = current_production_line;

current_location = 'abfss://' + current_project + '@adlsdpcom'+ current_environment +f'data.dfs.core.windows.net/' + current_production_line + '/'

if verbose_mode == 'debug':
    display("Debug Mode")
    display(f"current_division : {current_division}")
    display(f"current_environment : {current_environment}")
    display(f"current_project : {current_project}")
    display(f"current_production_line : {current_production_line}")
    display(f"current_catalog : {current_catalog}")
    display(f"current_schema : {current_schema}")
    display(f"current_location : {current_location}")

Source(s)

In [0]:
if current_environment =='preprd':
    source = f"""ext_mal_psql_maite_vision_board_test.public"""    
else:
    source = f"""ext_mal_psql_maite_vision_board_{current_environment}.public"""

In [0]:
source_gold = f"""mal_maite_common_{current_environment}.gold"""

In [0]:
source_bi = f"""mal_maite_{current_environment}"""

In [0]:
source_mal_maite = f"""mal_maite_{current_environment}"""

In [0]:
measurement = spark.table(f"""{source_gold}.measurements""")
parameters_variables_translations = spark.table(f"""{source}.parameters_variables_translations""")
parameters_variables = spark.table(f"""{source}.parameters_variables""")

fact_measurement

In [0]:
parameters_variables_translations_eng = parameters_variables_translations.filter((F.col("language") == 2) & (F.col("deleted") == False))
parameters_variables_deleted_false = parameters_variables.filter(F.col("deleted") == False)

In [0]:
measurement_array = (
    measurement
    .withColumn("value", col("measure.value").cast("float"))
    .withColumn("status", col("measure.status").cast("string"))
    .drop("measure")
)

In [0]:
df_measurement_1 = measurement_array.alias("a").join(
    parameters_variables_deleted_false.alias("b"),
    F.col("a.measure_name") == F.col("b.code"),
    "left"
).select(
    "a.*",
    "b.id_parameter_variable"
)

df_measurement = df_measurement_1.alias("a").join(
    parameters_variables_translations_eng.alias("b"),
    F.col("a.id_parameter_variable") == F.col("b.parameter_variable"),
    "left"
).select(
    "a.batch_id",
    "a.measure_name",
    "a.value",
    "a.status",
    "b.label"
)

df_measurement = df_measurement.filter(F.col("batch_id").isNotNull())

Import fact_measurement

In [0]:
current_process="fact_measurement"

In [0]:
target_fact_measurement = current_catalog +"."+current_schema+"."+current_process
print(target_fact_measurement)

In [0]:
all_columns =  df_measurement.columns
display(all_columns)

In [0]:
# define the primary key 
primary_key = [    
    'batch_id',
    'measure_name']

additional_columns = get_additional_columns(all_columns, primary_key)

In [0]:
if verbose_mode == 'debug': 
    print(additional_columns)


handle_table_update(
    df_measurement, 
    target_fact_measurement, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

batch_report

In [0]:
batches = f"""
SELECT
  b.id_batch,
  b.batch_number AS batch,
  b.mes_number,
  b.fabrication_order_number,
  ppl.name AS site_de_production,
  ppt.code AS type_de_production,
  b.harvest AS annee_de_recolte,
  b.planned_date AS debut_de_production,
  rs.name AS cahier_des_charges,
  gs.code AS espece,
  gv.code AS variete

FROM {source}.batches b

LEFT JOIN {source}.plants_production_lines ppl ON ppl.id_plant_production_line = b.production_line
LEFT JOIN {source}.parameters_production_types ppt ON ppt.id_parameter_production_type = b.production_type
LEFT JOIN {source}.requirement_specifications rs ON rs.id_requirement_specification = b.requirement_specifications AND rs.deleted = false
LEFT JOIN {source}.goods_varieties gv ON gv.id_good_variety = b.variety
LEFT JOIN {source}.goods_species gs ON gs.id_good_specy = gv.specy

WHERE b.deleted = false
"""

df_batches = spark.sql(batches)
df_batches.createOrReplaceTempView("batches")

df_batches_date_format = df_batches.withColumn(
    "debut_de_production",
    date_format(col("debut_de_production"), "dd-MM-yyyy"))

df_batches_date_format.createOrReplaceTempView("df_batches_date_format")

# Méthode de définition des dates de fin de production
date_fin_production = f"""
SELECT b.id_batch, b.planned_date, 
        l.kiln_unload_end_date, 
        l.kiln_unload_start_date, 
        l.germ_unload_start_date, 
        l.steep_c1_o1_filling_start_date
FROM {source}.batches b
LEFT JOIN {source_gold}.localizations_pivot l ON b.id_batch = l.batch_id
"""

df_date_fin_production = spark.sql(date_fin_production)

# Calcul du champ fin_de_production selon la priorité
df_date_fin_production = df_date_fin_production.withColumn(
    "fin_de_production",
    when(col("kiln_unload_end_date").isNotNull(), col("kiln_unload_end_date"))
    .when(col("kiln_unload_start_date").isNotNull(), col("kiln_unload_start_date"))
    .when(col("germ_unload_start_date").isNotNull(), col("germ_unload_start_date") + expr("INTERVAL 2 DAYS"))
    .when(col("steep_c1_o1_filling_start_date").isNotNull(), col("steep_c1_o1_filling_start_date") + expr("INTERVAL 8 DAYS"))
    .otherwise(col("planned_date") + expr("INTERVAL 8 DAYS")))

# Convertir la colonne 'calculation_date' en date uniquement (sans les heures)
df_date_fin_production = df_date_fin_production.withColumn('fin_de_production', col('fin_de_production').cast('date'))


df_date_fin_production.createOrReplaceTempView("date_fin_production")

df_date_fin_production_date_format = df_date_fin_production.withColumn(
    "fin_de_production",
    date_format(col("fin_de_production"), "dd-MM-yyyy"))

df_date_fin_production_date_format.createOrReplaceTempView("df_date_fin_production_date_format")

# Jointure pour ajouter les dates de fin de production
main = f"""
SELECT b.*, dfp.fin_de_production
FROM df_batches_date_format b
LEFT JOIN df_date_fin_production_date_format dfp ON b.id_batch = dfp.id_batch
"""
df_main = spark.sql(main)

# Conversion de toutes les colonnes en STRING
df_main = df_main.select([col(c).cast("STRING").alias(c) for c in df_main.columns])

# Liste des colonnes à dépivoter (toutes sauf "id_batch" et "batch")
columns_to_unpivot = [col for col in df_main.columns if col not in ["id_batch", "batch"]]

stack_expr = ", ".join([f"'{col}', {col}" for col in columns_to_unpivot])

df_main = df_main.selectExpr("id_batch", "batch", f"stack({len(columns_to_unpivot)}, {stack_expr}) as (parameter, value)").dropna(subset=["value"])

# Renomme les valeurs
df_main = df_main.withColumn(
    "parameter",
    when(col("parameter") == "fin_de_production", "End of production")
    .when(col("parameter") == "mes_number", "Mes number")
    .when(col("parameter") == "debut_de_production", "Start of production")
    .when(col("parameter") == "fabrication_order_number", "Fabrication order")
    .when(col("parameter") == "site_de_production", "Production line")
    .when(col("parameter") == "type_de_production", "Production type")
    .when(col("parameter") == "annee_de_recolte", "Harvest")
    .when(col("parameter") == "cahier_des_charges", "Specifications")
    .when(col("parameter") == "espece", "Species")
    .when(col("parameter") == "variete", "Variety")
    .otherwise(col("parameter"))
)

In [0]:
parameter_goods_malt = f"""
SELECT b.id_batch, b.batch_number AS batch, pvt.label AS parameter, me.value
FROM {source}.batches b

JOIN {source}.manual_entries me ON me.batch = b.id_batch AND me.deleted = false

JOIN {source}.parameters_variables pv ON pv.id_parameter_variable = me.parameter AND pv.deleted = false

JOIN {source}.parameters_variables_translations pvt ON pvt.parameter_variable = pv.id_parameter_variable AND pvt.language = 2 AND pvt.deleted = false

WHERE b.deleted = false AND pv.code IN ('malt_yield_r2', 'goods_weight', 'goods_moisture', 'malt_weight')
"""

df_parameter_goods_malt = spark.sql(parameter_goods_malt)


# Ajout malt_moisture via table gold
malt_moisture_gold = f"""
SELECT b.id_batch, b.batch_number AS batch, mqr.malt_moisture
FROM {source_gold}.malt_quality_results mqr
JOIN {source}.batches b ON b.fabrication_order_number = mqr.fabrication_order AND b.deleted = false
"""
df_malt_moisture_gold = spark.sql(malt_moisture_gold)

df_malt_moisture_gold = df_malt_moisture_gold.withColumn("parameter", lit("Moisture Malt")) \
                                  .withColumnRenamed("malt_moisture", "value") \
                                  .select("id_batch", "batch", "parameter", "value")

# Union des deux dataframes
df_parameter_goods_malt = df_parameter_goods_malt.unionByName(df_malt_moisture_gold)

df_parameter_goods_malt = df_parameter_goods_malt.withColumn("value", round("value", 2))

# Conversion de toutes les colonnes en STRING
df_parameter_goods_malt = df_parameter_goods_malt.select([col(c).cast("STRING").alias(c) for c in df_parameter_goods_malt.columns])

In [0]:
r2_sec = f"""
SELECT b.id_batch, b.batch_number AS batch, mqr.malt_moisture, pv.code AS parameter, me.value
FROM {source}.batches b
JOIN {source}.manual_entries me ON me.batch = b.id_batch AND me.deleted = false
JOIN {source}.parameters_variables pv ON pv.id_parameter_variable = me.parameter AND pv.deleted = false
LEFT JOIN {source_gold}.malt_quality_results mqr ON mqr.fabrication_order = b.fabrication_order_number
WHERE b.deleted = false AND pv.code IN ('malt_yield_r2', 'goods_weight', 'goods_moisture', 'malt_weight')
"""

df_r2_sec = spark.sql(r2_sec)

# Pivot
df_r2_sec = df_r2_sec.groupBy("id_batch", "batch", "malt_moisture").pivot("parameter").agg(F.first("value"))

# # Calcul du r2_sec
# df_r2_sec = df_r2_sec.withColumn("R2 Dry",round(
#         (col("malt_weight") - (col("malt_weight") * (col("malt_moisture") / 100))) /
#         (col("goods_weight") - (col("goods_weight") * (col("goods_moisture") / 100))) * 100, 1))

# Calcul du r2_sec (division sécurisée pour éviter DIVIDE_BY_ZERO)
denominateur = col("goods_weight") - (col("goods_weight") * (col("goods_moisture") / 100))
numerateur = col("malt_weight") - (col("malt_weight") * (col("malt_moisture") / 100))
df_r2_sec = df_r2_sec.withColumn("R2 Dry", round(
        when(denominateur != 0, numerateur / denominateur * 100).otherwise(None), 1))


# Garde seulement les colonnes 'id_batch', 'batch' et 'R2 Dry'
df_r2_sec = df_r2_sec.select("id_batch", "batch", 'R2 Dry')

# Dépivoter la colonne 'R2 Dry'
df_r2_sec = df_r2_sec.selectExpr("id_batch", "batch", "'R2 Dry' as parameter", "`R2 Dry` as value").dropna(subset=["value"])

# Conversion de toutes les colonnes en STRING
df_r2_sec = df_r2_sec.select([col(c).cast("STRING").alias(c) for c in df_r2_sec.columns])

In [0]:
parameter_categ_1 = f"""
SELECT b.id_batch, b.batch_number AS batch, pvt.label AS parameter, me.value

FROM {source}.batches b

JOIN {source}.manual_entries me ON me.batch = b.id_batch
    AND me.deleted = false

JOIN {source}.parameters_variables pv ON pv.id_parameter_variable = me.parameter
    AND pv.category = 1 AND pv.deleted = false

JOIN {source}.parameters_variables_translations pvt ON pvt.parameter_variable = pv.id_parameter_variable
    AND pvt.language = 2 AND pvt.deleted = false

WHERE b.deleted = false AND pv.code NOT IN ('goods_broken_grains_percent', 'goods_bad_grains_percent')
"""

df_parameter_categ_1 = spark.sql(parameter_categ_1)

df_parameter_categ_1 = df_parameter_categ_1.withColumn("value", round("value", 2))

# Supprime les erreurs
df_parameter_categ_1 = df_parameter_categ_1.filter(~((col("parameter") == "Date de calibrage") & (col("value") < 20230101)))

# Conversion en date pour les valeurs de 'Date calibrage'
df_parameter_categ_1 = df_parameter_categ_1.withColumn(
    "value",
    when(
        col("parameter") == "Date de calibrage",
        date_format(
            to_date(format_string("%08d", col("value").cast("long")), "yyyyMMdd"),
            "yyyy-MM-dd"
        )
    ).otherwise(col("value").cast("string"))
)

In [0]:
manual_entries = f"""
SELECT b.id_batch, b.batch_number AS batch, pvt.label AS parameter, me.value
FROM {source}.batches b

JOIN {source}.manual_entries me ON me.batch = b.id_batch AND me.deleted = false

JOIN {source}.parameters_variables pv ON pv.id_parameter_variable = me.parameter
    AND pv.category = 2

JOIN {source}.parameters_variables_translations pvt ON pvt.parameter_variable = pv.id_parameter_variable
    AND pvt.language = 2 AND pvt.deleted = false

WHERE b.deleted = false
"""

df_manual_entries = spark.sql(manual_entries)

df_manual_entries = df_manual_entries.withColumn("value", round("value", 2))

# Conversion de toutes les colonnes en STRING
df_manual_entries = df_manual_entries.select([col(c).cast("STRING").alias(c) for c in df_manual_entries.columns])

In [0]:
quality_results = f"""
SELECT b.id_batch, b.batch_number AS batch, gqr.goods_bad_grains_percent, gqr.goods_broken_grains_percent, mqr.malt_color_ebc, mqr.malt_free_amino_nitrogen, mqr.malt_friability, mqr.malt_moisture AS malt_moisture_quality, mqr.malt_soluble_beta_glucan, mqr.malt_viscosity, mqr.malt_soluble_protein, mqr.malt_broken_kernels, mqr.malt_turbidity_ebc, mqr.malt_partly_unmodified_grains, mqr.malt_fine_extract, mqr.malt_ph, mqr.malt_kolbach_index, mqr.malt_boiled_wort_colour_ebc, mqr.malt_hartong_45, mqr.malt_protein_content, mqr.malt_whole_grains
FROM {source}.batches b
LEFT JOIN {source_gold}.goods_quality_results gqr ON gqr.fabrication_order = b.fabrication_order_number
LEFT JOIN {source_gold}.malt_quality_results mqr ON mqr.fabrication_order = b.fabrication_order_number
WHERE b.deleted = false
"""

df_quality_results = spark.sql(quality_results)

# Liste des colonnes à dépivoter (toutes sauf "id_batch" et "batch")
columns_to_unpivot = [col for col in df_quality_results.columns if col not in ["id_batch", "batch"]]

stack_expr = ", ".join([f"'{col}', {col}" for col in columns_to_unpivot])

df_quality_results = df_quality_results.selectExpr("id_batch", "batch", f"stack({len(columns_to_unpivot)}, {stack_expr}) as (parameter, value)").dropna(subset=["value"])

# Renomme les valeurs
df_quality_results = df_quality_results.withColumn(
    "parameter",
    when(col("parameter") == "malt_moisture_quality", "Moisture Malt Quality")
    .when(col("parameter") == "malt_soluble_beta_glucan", "malt_soluble_betaglucan")
    .otherwise(col("parameter"))
)

df_quality_results = df_quality_results.withColumn("value", round("value", 2))

df_quality_results.createOrReplaceTempView("quality_results")

quality = f"""
SELECT qr.*, pet.label, pvt.label AS label_2
FROM quality_results qr
LEFT JOIN {source}.parameters_evaluations pe ON pe.code = qr.parameter AND pe.deleted = false
LEFT JOIN {source}.parameters_evaluations_translations pet ON pet.id_parameter_evaluation = pe.id_parameter_evaluation AND pet.language = 2 AND pet.deleted = false
LEFT JOIN {source}.parameters_variables pv ON pv.code = qr.parameter AND pv.deleted = false
LEFT JOIN {source}.parameters_variables_translations pvt ON pvt.parameter_variable = pv.id_parameter_variable AND pvt.language = 2 AND pvt.deleted = false
"""

df_quality = spark.sql(quality)

df_quality = df_quality.withColumn(
    "parameter_label",
    when(col("label_2").isNotNull(), col("label_2"))
    .when(col("label").isNotNull(), col("label"))
    .otherwise(col("parameter"))).drop("label", "label_2")

# Conversion de toutes les colonnes en STRING
df_quality = df_quality.select([col(c).cast("STRING").alias(c) for c in df_quality.columns])

df_quality = df_quality.select(
    "id_batch",
    "batch",
    F.col("parameter_label").alias("parameter"),
    "value"
)

In [0]:
bqt = f"""
SELECT b.fabrication_order_number, pe.code, btq.upper_limit, btq.lower_limit
FROM {source}.batches_quality_targets btq
JOIN {source}.batches b ON b.id_batch = btq.batch AND b.deleted = false
JOIN {source}.parameters_evaluations pe ON btq.evaluation_parameter = pe.id_parameter_evaluation
WHERE btq.deleted = false AND b.fabrication_order_number IS NOT NULL
"""

df_bqt = spark.sql(bqt)

# Max upper_limit
df_bqt_max = df_bqt.groupBy("fabrication_order_number").pivot("code").agg(F.max("upper_limit"))

# Renommer les colonnes dans df_bqt_max
df_bqt_max = df_bqt_max.withColumnRenamed('malt_color_ebc', 'coloration_EBC_max') \
                       .withColumnRenamed('malt_free_amino_nitrogen', 'FAN_max') \
                       .withColumnRenamed('malt_friability', 'friabilite_max') \
                       .withColumnRenamed('malt_soluble_betaglucan', 'betaG_max') \
                       .withColumnRenamed('product_moisture', 'hum_malt_max')

# Max lower_limit
df_bqt_min = df_bqt.groupBy("fabrication_order_number").pivot("code").agg(F.max("lower_limit"))

# Renommer les colonnes dans df_bqt_min
df_bqt_min = df_bqt_min.withColumnRenamed('malt_color_ebc', 'coloration_EBC_min') \
                       .withColumnRenamed('malt_free_amino_nitrogen', 'FAN_min') \
                       .withColumnRenamed('malt_friability', 'friabilite_min') \
                       .withColumnRenamed('malt_soluble_betaglucan', 'betaG_min') \
                       .withColumnRenamed('product_moisture', 'hum_malt_min')

# Jointure
df_bqt = df_bqt_max.join(df_bqt_min.select('fabrication_order_number', 'coloration_EBC_min', 'FAN_min', 'friabilite_min', 'betaG_min', 'hum_malt_min'), on='fabrication_order_number', how='left')

# Sélection des colonnes finales
df_bqt = df_bqt.select(
    'fabrication_order_number',
    'FAN_max',
    'FAN_min',
    'hum_malt_max',
    'hum_malt_min',
    'betaG_max',
    'betaG_min',
    'friabilite_max',
    'friabilite_min',
    'coloration_EBC_max',
    'coloration_EBC_min'
)

# Créer une vue temporaire pour le DataFrame
df_bqt.createOrReplaceTempView("batches_quality_targets_min_max")

indice_qualite = f"""
SELECT
    b.id_batch,
    b.batch_number AS batch,
    malt_free_amino_nitrogen AS FAN,
    malt_soluble_beta_glucan AS betaG,
    malt_friability AS friabilite,
    malt_moisture AS hum_malt,
    malt_color_ebc AS coloration_EBC,
    bqtmm.*
FROM {source}.batches b
JOIN {source_gold}.malt_quality_results mqr ON mqr.fabrication_order = b.fabrication_order_number
JOIN batches_quality_targets_min_max bqtmm ON bqtmm.fabrication_order_number = mqr.fabrication_order
"""

df_indice_qualite = spark.sql(indice_qualite)

# Si dans les bornes alors = 0,2 sinon 0 puis somme finale des 5 critères 
df_indice_qualite = df_indice_qualite.withColumn('FAN_result', F.when((F.col('FAN_min') <= F.col('FAN')) & (F.col('FAN') <= F.col('FAN_max')), 20).otherwise(0))
df_indice_qualite = df_indice_qualite.withColumn('hum_malt_result', F.when((F.col('hum_malt_min') <= F.col('hum_malt')) & (F.col('hum_malt') <= F.col('hum_malt_max')), 20).otherwise(0))
df_indice_qualite = df_indice_qualite.withColumn('betaG_result', F.when((F.col('betaG_min') <= F.col('betaG')) & (F.col('betaG') <= F.col('betaG_max')), 20).otherwise(0))
df_indice_qualite = df_indice_qualite.withColumn('friabilite_result', F.when((F.col('friabilite_min') <= F.col('friabilite')) & (F.col('friabilite') <= F.col('friabilite_max')), 20).otherwise(0))
df_indice_qualite = df_indice_qualite.withColumn('coloration_EBC_result', F.when((F.col('coloration_EBC_min') <= F.col('coloration_EBC')) & (F.col('coloration_EBC') <= F.col('coloration_EBC_max')), 20).otherwise(0))
df_indice_qualite = df_indice_qualite.withColumn('Indice qualite', F.round(F.col('FAN_result') + F.col('hum_malt_result') + F.col('betaG_result') + F.col('friabilite_result') + F.col('coloration_EBC_result'),1))

df_indice_qualite = df_indice_qualite.select('id_batch', 'batch', 'Indice qualite')

# Dépivoter la colonne 'Indice qualite'
df_indice_qualite = df_indice_qualite.selectExpr("id_batch", "batch", "'Indice qualite' as parameter", "`Indice qualite` as value").dropna(subset=["value"])

# Conversion de toutes les colonnes en STRING
df_indice_qualite = df_indice_qualite.select([col(c).cast("STRING").alias(c) for c in df_quality.columns])

# Ajoute % derrière chaque valeur
df_indice_qualite = df_indice_qualite.withColumn("value",concat(col("value").cast("string"), lit("%")))

In [0]:
# Concatenation des DataFrames
df_batch_report = df_main.unionByName(df_parameter_goods_malt) \
                   .unionByName(df_r2_sec) \
                   .unionByName(df_parameter_categ_1) \
                   .unionByName(df_manual_entries) \
                   .unionByName(df_quality) \
                   .unionByName(df_indice_qualite)

# Supprime les lignes vident dans la colonne 'value'
df_batch_report = df_batch_report.filter(df_batch_report["value"].isNotNull())

# Remplace les '.' par ',' dans la colonne 'value'
df_batch_report = df_batch_report.withColumn("value", regexp_replace(col("value"), r"\.", ","))

# Supprime les doublons
df_batch_report = df_batch_report.dropDuplicates(['id_batch', 'parameter'])

In [0]:
# Supprime les colonnes suivantes --> modif suite demande William réorganisation des colonnes
to_drop = [
    "Mes number",
    "Moisture Malt",
    "Indice qualite",
    "#_volume",
    "Goods Calibration Date",
    "Goods Storage Duration",
    ]

df_batch_report = df_batch_report.filter(col("parameter").isNull() | (~trim(col("parameter")).isin(to_drop)))

In [0]:
# Colonne 'custom_order'
df_batch_report = df_batch_report.withColumn(
    "custom_order",
    when(trim(col("parameter")) == "Fabrication order", 1)
    .when(trim(col("parameter")) == "Production line", 2)
    .when(trim(col("parameter")) == "Production type", 3)
    .when(trim(col("parameter")) == "Harvest", 4)
    .when(trim(col("parameter")) == "Start of production", 5)
    .when(trim(col("parameter")) == "End of production", 6)
    .when(trim(col("parameter")) == "Specifications", 7)
    .when(trim(col("parameter")) == "Species", 8)
    .when(trim(col("parameter")) == "Variety", 9)
    .when(trim(col("parameter")) == "Malt Weight", 10)
    .when(trim(col("parameter")) == "Malt Yield R2", 11)
    .when(trim(col("parameter")) == "R2 Dry", 12)
    .when(trim(col("parameter")) == "Moisture Malt Quality", 13)
    .when(trim(col("parameter")) == "Friability", 14)
    .when(trim(col("parameter")) == "Color EBC", 15)
    .when(trim(col("parameter")) == "FAN", 16)
    .when(trim(col("parameter")) == "Betaglucans", 17)
    .when(trim(col("parameter")) == "Viscosity", 18)
    .when(trim(col("parameter")) == "SOLUBLE PROTEIN", 19)
    .when(trim(col("parameter")) == "BROKEN KERNELS", 20)
    .when(trim(col("parameter")) == "TURBIDITY EBC", 21)
    .when(trim(col("parameter")) == "PARTLY UNMODIFIED GRAINS", 22)
    .when(trim(col("parameter")) == "FINE EXTRACT (DM)", 23)
    .when(trim(col("parameter")) == "PH", 24)
    .when(trim(col("parameter")) == "KOLBACH INDEX", 25)
    .when(trim(col("parameter")) == "BOILED WORT COLOUR EBC", 26)
    .when(trim(col("parameter")) == "HARTONG 45", 27)
    .when(trim(col("parameter")) == "malt_protein_content", 28)
    .when(trim(col("parameter")) == "malt_whole_grains", 29)
    .when(trim(col("parameter")) == "Goods Weight", 30)
    .when(trim(col("parameter")) == "Goods Moisture", 31)
    .when(trim(col("parameter")) == "Goods Protein", 32)
    .when(trim(col("parameter")) == "GL 4mL 72h", 33) # Pas de traduction en ENG
    .when(trim(col("parameter")) == "GL 8mL 72h", 34) # Pas de traduction en ENG
    .when(trim(col("parameter")) == "Goods Broken Grains Percent", 35)
    .when(trim(col("parameter")) == "Goods Bad Grains Percent", 36)
    .when(trim(col("parameter")) == "#_mass", 37)
    .when(trim(col("parameter")) == "#_doses", 38)
    .when(trim(col("parameter")) == "Steep C1 Moisture", 39)
    .when(trim(col("parameter")) == "Steep C2 Moisture", 40)
    .when(trim(col("parameter")) == "Pregerm Moisture", 41)
    .when(trim(col("parameter")) == "Sprouted grains St2", 42)
    .when(trim(col("parameter")) == "Germ C1 Moisture", 43)
    .when(trim(col("parameter")) == "Sprouted grains D1", 44)
    .when(trim(col("parameter")) == "Germ 2 - grains piqués manquants", 45) # Pas de traduction en ENG
    .when(trim(col("parameter")) == "Germ C2 Moisture", 46)
    .when(trim(col("parameter")) == "Germ C3 Moisture", 47)
    .when(trim(col("parameter")) == "Germ C4 Moisture", 48)
    .when(trim(col("parameter")) == "Kiln Load Moisture", 49)
    .otherwise(99)
)

Import fact batch report

In [0]:
current_process="fact_batch_report"

In [0]:
target_fact_batch_report = current_catalog +"."+current_schema+"."+current_process
print(target_fact_batch_report)

In [0]:
all_columns =  df_batch_report.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_batch'
    ,'parameter']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_batch_report, 
    target_fact_batch_report, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

fact_weather

In [0]:
# Liste des tables à fusionner
tables = [
    f"{source_bi}.rouen1.master_table",
    f"{source_bi}.nogent1.master_table",
    f"{source_bi}.nogent2.master_table",
    f"{source_bi}.polisy1.master_table",
    f"{source_bi}.prouvy1.master_table",
    f"{source_bi}.strasbourg2.master_table",
    f"{source_bi}.buzau1.master_table",
    f"{source_bi}.bolelemi1.master_table"
]

# Liste des colonnes à sélectionner
colonnes = [
    "batch_id",
    "fabrication_order",
    "production_planned_date",
    "steep_c1_start_date", "steep_c1_weather_temperature", "steep_c1_weather_hygrometry",
    "steep_c2_start_date", "steep_c2_weather_temperature", "steep_c2_weather_hygrometry",
    "germ_c1_start_date", "germ_c1_weather_temperature", "germ_c1_weather_hygrometry",
    "germ_c2_start_date", "germ_c2_weather_temperature", "germ_c2_weather_hygrometry",
    "germ_c3_start_date", "germ_c3_weather_temperature", "germ_c3_weather_hygrometry",
    "germ_c4_start_date", "germ_c4_weather_temperature", "germ_c4_weather_hygrometry",
    "germ_c5_start_date", "germ_c5_weather_temperature", "germ_c5_weather_hygrometry",
    "kiln_c1_start_date", "kiln_c1_weather_temperature", "kiln_c1_weather_hygrometry",
    "kiln_c2_start_date", "kiln_c2_weather_temperature", "kiln_c2_weather_hygrometry"
]

# Initialisation du DataFrame final
df_weather_realized = None

# Boucle pour lire et fusionner les tables
for table in tables:
    df = spark.table(table)
    df = df.withColumn("production_line", F.upper(F.col("production_line")))
    df = df.select("production_line", *colonnes)

    if df_weather_realized is None:
        df_weather_realized = df
    else:
        df_weather_realized = df_weather_realized.unionByName(df)

In [0]:
weather_predicted = f"""
SELECT r.batch AS batch_id, pv.code AS parameter, rp.realized_value
FROM {source}.recommendations_production rp
JOIN {source}.parameters_variables pv ON pv.id_parameter_variable = rp.parameter AND pv.deleted = false
JOIN {source}.recommendations r ON r.id_recommendation = rp.recommendation AND r.deleted = false

JOIN (
  SELECT r.batch AS batch_id, pv.code AS parameter, MAX(rp.created_at) AS max_created_at
  FROM {source}.recommendations_production rp
  JOIN {source}.parameters_variables pv ON pv.id_parameter_variable = rp.parameter AND pv.deleted = false
  JOIN {source}.recommendations r ON r.id_recommendation = rp.recommendation AND r.deleted = false
  WHERE rp.deleted = false AND pv.code IN ('steep_c1_weather_temperature', 'steep_c1_weather_hygrometry', 'steep_c2_weather_temperature', 'steep_c2_weather_hygrometry', 'germ_c1_weather_temperature', 'germ_c1_weather_hygrometry', 'germ_c2_weather_temperature', 'germ_c2_weather_hygrometry', 'germ_c3_weather_temperature', 'germ_c3_weather_hygrometry', 'germ_c4_weather_temperature', 'germ_c4_weather_hygrometry', 'germ_c5_weather_temperature', 'germ_c5_weather_hygrometry', 'kiln_c1_weather_temperature', 'kiln_c1_weather_hygrometry', 'kiln_c2_weather_temperature', 'kiln_c2_weather_hygrometry')
  GROUP BY r.batch, pv.code
) a ON a.max_created_at = rp.created_at AND a.parameter = pv.code AND a.batch_id = r.batch

WHERE rp.deleted = false AND pv.code IN ('steep_c1_weather_temperature', 'steep_c1_weather_hygrometry', 'steep_c2_weather_temperature', 'steep_c2_weather_hygrometry', 'germ_c1_weather_temperature', 'germ_c1_weather_hygrometry', 'germ_c2_weather_temperature', 'germ_c2_weather_hygrometry', 'germ_c3_weather_temperature', 'germ_c3_weather_hygrometry', 'germ_c4_weather_temperature', 'germ_c4_weather_hygrometry', 'germ_c5_weather_temperature', 'germ_c5_weather_hygrometry', 'kiln_c1_weather_temperature', 'kiln_c1_weather_hygrometry', 'kiln_c2_weather_temperature', 'kiln_c2_weather_hygrometry')
"""

df_weather_predicted = spark.sql(weather_predicted)

# Arrondir la colonne realized_value à 1 décimal
df_weather_predicted = df_weather_predicted.withColumn("realized_value",F.col("realized_value").cast("double"))
df_weather_predicted = df_weather_predicted.withColumn("realized_value",F.round("realized_value", 1))

# Ajouter "_predicted" à chaque valeur de la colonne parameter
df_weather_predicted = df_weather_predicted.withColumn("parameter",F.concat(F.col("parameter"), F.lit("_predicted")))

# Pivot du DataFrame
df_weather_predicted = df_weather_predicted.groupBy("batch_id").pivot("parameter").agg(F.first("realized_value"))

In [0]:
# Merge entre les deux DataFrames
df_weather = df_weather_realized.join(df_weather_predicted,on="batch_id",how="left")

# Trie des colonnes
df_weather = df_weather.select("batch_id", "production_line", "fabrication_order", "production_planned_date",
                               "steep_c1_start_date", "steep_c1_weather_temperature", "steep_c1_weather_hygrometry", "steep_c1_weather_temperature_predicted", "steep_c1_weather_hygrometry_predicted",
                               "steep_c2_start_date", "steep_c2_weather_temperature", "steep_c2_weather_hygrometry", "steep_c2_weather_temperature_predicted", "steep_c2_weather_hygrometry_predicted",
                               "germ_c1_start_date", "germ_c1_weather_temperature", "germ_c1_weather_hygrometry", "germ_c1_weather_temperature_predicted", "germ_c1_weather_hygrometry_predicted",
                               "germ_c2_start_date", "germ_c2_weather_temperature", "germ_c2_weather_hygrometry", "germ_c2_weather_temperature_predicted", "germ_c2_weather_hygrometry_predicted",
                               "germ_c3_start_date", "germ_c3_weather_temperature", "germ_c3_weather_hygrometry", "germ_c3_weather_temperature_predicted", "germ_c3_weather_hygrometry_predicted",
                               "germ_c4_start_date", "germ_c4_weather_temperature", "germ_c4_weather_hygrometry", "germ_c4_weather_temperature_predicted", "germ_c4_weather_hygrometry_predicted",
                               "germ_c5_start_date", "germ_c5_weather_temperature", "germ_c5_weather_hygrometry", "germ_c5_weather_temperature_predicted", "germ_c5_weather_hygrometry_predicted",
                               "kiln_c1_start_date", "kiln_c1_weather_temperature", "kiln_c1_weather_hygrometry", "kiln_c1_weather_temperature_predicted", "kiln_c1_weather_hygrometry_predicted",
                               "kiln_c2_start_date", "kiln_c2_weather_temperature", "kiln_c2_weather_hygrometry", "kiln_c2_weather_temperature_predicted", "kiln_c2_weather_hygrometry_predicted")

# Renomme les colonnes 'batch_id'
df_weather = df_weather.withColumnRenamed("batch_id", "id_batch")

# Liste des colonnes à arrondir
columns_to_round = [
    "steep_c1_weather_temperature", "steep_c1_weather_hygrometry",
    "steep_c1_weather_temperature_predicted", "steep_c1_weather_hygrometry_predicted",
    "steep_c2_weather_temperature", "steep_c2_weather_hygrometry",
    "steep_c2_weather_temperature_predicted", "steep_c2_weather_hygrometry_predicted",
    "germ_c1_weather_temperature", "germ_c1_weather_hygrometry",
    "germ_c1_weather_temperature_predicted", "germ_c1_weather_hygrometry_predicted",
    "germ_c2_weather_temperature", "germ_c2_weather_hygrometry",
    "germ_c2_weather_temperature_predicted", "germ_c2_weather_hygrometry_predicted",
    "germ_c3_weather_temperature", "germ_c3_weather_hygrometry",
    "germ_c3_weather_temperature_predicted", "germ_c3_weather_hygrometry_predicted",
    "germ_c4_weather_temperature", "germ_c4_weather_hygrometry",
    "germ_c4_weather_temperature_predicted", "germ_c4_weather_hygrometry_predicted",
    "germ_c5_weather_temperature", "germ_c5_weather_hygrometry",
    "germ_c5_weather_temperature_predicted", "germ_c5_weather_hygrometry_predicted",
    "kiln_c1_weather_temperature", "kiln_c1_weather_hygrometry",
    "kiln_c1_weather_temperature_predicted", "kiln_c1_weather_hygrometry_predicted",
    "kiln_c2_weather_temperature", "kiln_c2_weather_hygrometry",
    "kiln_c2_weather_temperature_predicted", "kiln_c2_weather_hygrometry_predicted"
]

# Arrondir les colonnes à 1 décimale
for column in columns_to_round:
    df_weather = df_weather.withColumn(column, F.round(F.col(column), 1))

Import fact_weather

In [0]:
current_process="fact_weather"

In [0]:
target_fact_weather = current_catalog +"."+current_schema+"."+current_process
print(target_fact_weather)

In [0]:
all_columns =  df_weather.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_batch']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_weather, 
    target_fact_weather, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

fact_energy

In [0]:
energy = f"""
SELECT me.batch AS id_batch, me.parameter AS id_parameter, me.value
FROM {source}.manual_entries me
JOIN {source}.parameters_variables pv ON pv.id_parameter_variable = me.parameter AND pv.deleted = false
WHERE me.deleted = false AND pv.category = 4
"""

df_energy = spark.sql(energy)

Import fact_energy

In [0]:
current_process="fact_energy"

In [0]:
target_fact_energy = current_catalog +"."+current_schema+"."+current_process
print(target_fact_energy)

In [0]:
all_columns =  df_energy.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_batch'
    ,'id_parameter']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_energy, 
    target_fact_energy, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )

dim_batch

In [0]:
dim_batch = f"""
SELECT
    b.id_batch,
    b.batch_number,
    b.mes_number,
    b.fabrication_order_number AS ordre_fabrication,
    b.harvest AS annee_recolte,
    b.production_line AS id_site,
    b.requirement_specifications AS id_cdc,
    b.production_type AS id_type,
    b.variety AS id_variete,
    b.planned_date AS debut_de_production,
    dfp.fin_de_production
FROM {source}.batches b
LEFT JOIN date_fin_production dfp ON b.id_batch = dfp.id_batch
WHERE b.deleted = false
"""

df_dim_batch = spark.sql(dim_batch)

Import dim_batch

In [0]:
current_process="dim_batch"

In [0]:
target_dim_batch = current_catalog +"."+current_schema+"."+current_process
print(target_dim_batch)

In [0]:
all_columns =  df_dim_batch.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_batch']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_dim_batch, 
    target_dim_batch, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )